# Conclusiones

## 22. Replicar Ridge Baseline para vol_14 y vol_21

Se entrena Ridge con las mismas features enriquecidas para los targets de 14 y 21 días, con el shift corregido.

In [ ]:
def evaluar_ridge_target(df, feature_cols, target_col, label):
    df_m = df.dropna(subset=feature_cols + [target_col]).copy()
    df_m = df_m.sort_values("date").reset_index(drop=True)
    cut = int(len(df_m) * 0.75)
    train = df_m.iloc[:cut]; test = df_m.iloc[cut:]

    pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])
    pipe.fit(train[feature_cols], train[target_col])
    yp_test = pipe.predict(test[feature_cols])

    r2 = r2_score(test[target_col], yp_test)
    rmse = np.sqrt(mean_squared_error(test[target_col], yp_test))
    mae = mean_absolute_error(test[target_col], yp_test)

    # Gráfico
    plt.figure(figsize=(14, 5))
    plt.plot(test["date"].values, test[target_col].values, label="Real")
    plt.plot(test["date"].values, yp_test, label="Predicho", linestyle="--")
    plt.title(f"Ridge — {label} | R²={r2:.4f} | RMSE={rmse:.6f}")
    plt.xlabel("Fecha"); plt.ylabel("Volatilidad"); plt.legend(); plt.grid(True, alpha=0.3)
    plt.show()

    return {"Target": label, "RMSE": rmse, "MAE": mae, "R2": r2}

# Targets adicionales
df["target_vol_14"] = df["vol_14"].shift(-14)
df["target_vol_21"] = df["vol_21"].shift(-21)

r_7 = evaluar_ridge_target(df, feature_cols, "target_vol_7", "vol_7 (shift -7)")
r_14 = evaluar_ridge_target(df, feature_cols, "target_vol_14", "vol_14 (shift -14)")
r_21 = evaluar_ridge_target(df, feature_cols, "target_vol_21", "vol_21 (shift -21)")

print("\n=== Comparación Ridge por ventana de volatilidad ===")
print(pd.DataFrame([r_7, r_14, r_21]).to_string(index=False))

---
> **Interpretación:** Ridge se replica para los horizontes de 14 y 21 días con el mismo pipeline y las mismas 52 features. Se espera que el R² disminuya a medida que aumenta el horizonte de predicción — predecir volatilidad a 21 días es más difícil que a 7 días, ya que hay más incertidumbre acumulada. Esta réplica valida que el enfoque metodológico es consistente independientemente de la ventana objetivo.

## 23. Conclusiones

### Hallazgos principales

1. **Corrección metodológica:** El cambio de shift(-1) a shift(-7) eliminó el overlap temporal que inflaba artificialmente el R² (de 0.71 a ~0.07). Esto demuestra la importancia de diseñar targets sin fuga de información.

2. **Modelos lineales vs no lineales:** Ridge fue el mejor modelo de regresión en test, mientras que los modelos no lineales (RF, XGBoost, SVR) sufrieron overfitting severo. Esto sugiere que la señal predictiva disponible es mayormente lineal con las features actuales.

3. **Clasificación:** Los modelos de clasificación permiten abordar el problema desde otra perspectiva (¿subirá la volatilidad?), ofreciendo métricas complementarias como AUC y F1.

4. **Optimización:** Los métodos bayesianos (Optuna) y genéticos (DEAP) demostraron ser más eficientes que Grid Search para explorar espacios de hiperparámetros de alta dimensión.

5. **Comparación estadística:** Los tests de Diebold-Mariano y DeLong permiten validar si las diferencias entre modelos son estadísticamente significativas, no solo numéricas.

